# Structured Output Improvements

https://developers.openai.com/api/docs/guides/structured-outputs

In [1]:
import json

from textwrap import dedent
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

client = OpenAI()

MODEL = "gpt-5.4-mini"

JOB_DESCRIPTION = dedent("""
    We are looking for a Software Engineer to join our team and help us build the next generation of AI-powered features in our product. 
    The ideal candidate has experience with Python and LLM APUs.
""")

CLASSIFY_INSTRUCTIONS = dedent("""
    You classify whether a job posting is truly for an AI Engineering role.

    AI Engineering definition:
    - AI engineering means building applications on top of foundation models or integrating them into products.
    - Traditional ML engineering focuses on building, training, or tuning models.
    - MLOps and platform engineering focus on infrastructure and tooling, not AI-powered product features.

    Decision rules:
    - Set is_ai_engineering_role to true when the main responsibility is building product or application features on top of foundation models or LLMs.
    - Set is_ai_engineering_role to false when the role is mainly traditional software engineering, data science, analytics, ML research, model training, classical ML engineering, MLOps, or platform work.
    - If the posting is ambiguous or unclear, set is_ai_engineering_role to false.
    - In reason, briefly explain the main evidence for the decision in one sentence.
""")

## Approach 1: Raw JSON schema

We describe the response shape by writing a JSON schema by hand. This works, but it has a few pain points:

- **Boilerplate:** the schema is verbose. We always have to mark every property as `required` and set `additionalProperties` to `false`. This gets hard to read once the schema grows.
- **No single source of truth:** the schema and the code that reads the result live in two separate places. Rename a property in one and forget the other, and they drift apart.
- **Plain dictionary:** the SDK returns the result as text, which we load into a dict. A typo in a key like `classification.get("is_ai_engineering_role")` silently returns nothing instead of raising an error, and we get no autocompletion.

In [ ]:
CLASSIFY_SCHEMA = {
    "type": "object",
    "properties": {
        "is_ai_engineering_role": {"type": "boolean"},
        "reason": {"type": "string"},
    },
    "required": ["is_ai_engineering_role", "reason"],
    "additionalProperties": False,
}

In [3]:
response = client.responses.create(
    model=MODEL,
    instructions=CLASSIFY_INSTRUCTIONS,
    input=JOB_DESCRIPTION,
    text={
        "format": {
            "type": "json_schema",
            "name": "job_classification",
            "schema": CLASSIFY_SCHEMA,
            "strict": True,
        },
        "verbosity": "low",
    },
)

# classification = dictionary
classification = json.loads(response.output_text)

print("Is AI Engineering Role:", classification.get("is_ai_engineering_role"))
print("Reason:", classification.get("reason"))

Is AI Engineering Role: True
Reason: The role explicitly focuses on building AI-powered product features and mentions LLM APIs, which matches AI engineering on top of foundation models.


## Approach 2: Pydantic

[Pydantic](https://docs.pydantic.dev/latest/) is the most widely used data validation library for Python. We define a Pydantic model, which is like a data class but with extra functionality, and pass it straight to the OpenAI SDK.

Instead of `client.responses.create`, we call `client.responses.parse` and hand our model to the `text_format` parameter. The SDK then returns a real instance of our model in `response.output_parsed`, not a plain dictionary.

In [4]:
from pydantic import BaseModel

class JobClassification(BaseModel):
    is_ai_engineering_role: bool
    reason: str

In [5]:
response = client.responses.parse(
    model=MODEL,
    instructions=CLASSIFY_INSTRUCTIONS,
    input=JOB_DESCRIPTION,
    text_format=JobClassification
)

job_classification = response.output_parsed
if job_classification is not None:
    print(job_classification.is_ai_engineering_role)
    print(job_classification.reason)

True
The posting focuses on building AI-powered product features and working with LLM APIs, which fits AI engineering on top of foundation models.


### Why check for `None`?

`response.output_parsed` is typed as `JobClassification | None`, so it is not always guaranteed to be our model. The model can fail to return a classification in some cases, for example when it refuses for safety reasons, when our API credits are exhausted, or when it wants to make a tool call first. That is why we check `if job_classification is not None` before reading its properties.

## Three advantages of the Pydantic approach

1. **Less boilerplate:** we define a Pydantic model instead of a raw JSON schema.
2. **Single source of truth:** to change the data model, we edit one place, the Pydantic model.
3. **A real Python object:** the SDK returns an actual object instead of a plain dictionary, so we get type safety and autocompletion in the IDE.